In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("schema_prefix", "elina_sharabura")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA_PREFIX = dbutils.widgets.get("schema_prefix")

SOURCE_TABLE = f"{CATALOG}.{SCHEMA_PREFIX}_bronze.products"
TARGET_TABLE = f"{CATALOG}.{SCHEMA_PREFIX}_silver.products"
QUARANTINE_TABLE = f"{CATALOG}.{SCHEMA_PREFIX}_silver.dq_quarantine"

TRACKED_COLUMNS = ["category", "name", "price_usd", "cost_usd", "margin_usd"]

In [0]:
import logging

logger = logging.getLogger("lab4.silver_products")
logger.setLevel(logging.INFO)
logger.propagate = False
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(name)s - %(message)s"))
    logger.addHandler(handler)

In [0]:
bronze_df = spark.table(SOURCE_TABLE)

##### Data quality rules

In [0]:
dq_rules = (
    F.col("product_id").isNotNull()
    & (F.col("price_usd") > 0)
    & (F.col("cost_usd") >= 0)
    & (F.abs(F.col("price_usd") - F.col("cost_usd") - F.col("margin_usd")) < 0.01)
    & F.col("_rescued_data").isNull()
)

is_valid = F.coalesce(dq_rules, F.lit(False))

valid_df = bronze_df.filter(is_valid)
invalid_df = bronze_df.filter(~is_valid)

##### Quarantine
##### Invalid rows are stored in `dq_quarantine`. MERGE prevents duplicates on re-runs.

In [0]:
quarantine_df = invalid_df.select(
    F.lit(SOURCE_TABLE).alias("source_table"),
    F.to_json(F.struct(*invalid_df.columns)).alias("record"),
    F.current_timestamp().alias("quarantined_at"),
)

quarantine_result = (
    DeltaTable.forName(spark, QUARANTINE_TABLE).alias("t")
    .merge(quarantine_df.alias("s"), "t.source_table = s.source_table AND t.record = s.record")
    .whenNotMatchedInsertAll()
    .execute()
    .first()
)
logger.info("Quarantined %d new invalid rows", quarantine_result["num_inserted_rows"])

In [0]:
latest_first = Window.partitionBy("product_id").orderBy(
    F.col("ingestion_timestamp").desc(),
    F.col("source_filename").desc(),
)

deduped_df = (
    valid_df
    .withColumn("row_num", F.row_number().over(latest_first))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

In [0]:
silver_df = deduped_df.select(
    "product_id",
    *TRACKED_COLUMNS,
    "source_filename",
    "ingestion_timestamp",
    F.current_timestamp().alias("processed_at"),
)

changed_condition = " OR ".join(f"NOT (t.{c} <=> s.{c})" for c in TRACKED_COLUMNS)

##### MERGE - SCD Type 1
##### New products are inserted, changed products are overwritten, unchanged products are skipped.

In [0]:
merge_result = (
    DeltaTable.forName(spark, TARGET_TABLE).alias("t")
    .merge(silver_df.alias("s"), "t.product_id = s.product_id")
    .whenMatchedUpdateAll(condition=changed_condition)
    .whenNotMatchedInsertAll()
    .execute()
    .first()
)
logger.info(
    "%s: %d inserted, %d updated",
    TARGET_TABLE,
    merge_result["num_inserted_rows"],
    merge_result["num_updated_rows"],
)